# Relatório Técnico e Laboratório Computacional: Projeto IA 25/26
### **Jogo Variant Connect-Four: PopOut**
**Autores:** Kirill Egorkin, Tomás Carvalhosa  
**Instituição:** Departamento de Ciência de Computadores, Faculdade de Ciências da Universidade do Porto (DCC/FCUP)  
**Contexto:** Unidade Curricular de Inteligência Artificial  

---

## Visão Geral do Notebook
Este Jupyter Notebook serve como um relatório dinâmico e laboratório computacional unificado do projeto. Ele descreve a evolução da arquitetura do nosso sistema de jogo híbrido, que combina **Pesquisa Heurística em Tempo Real (MCTS)** e **Modelos Preditivos Indutivos (Árvores de Decisão ID3)**, suportados por uma representação de dados de ultra-alta performance baseada em **Bitboards**.

---
## Capítulo 1: Engenharia de Baixo Nível - Estrutura do Tabuleiro via Bitboards

### 1.1 A Importância Crítica da Performance
O algoritmo Monte Carlo Tree Search (MCTS) baseia-se na simulação massiva de partidas aleatórias ou guiadas (*rollouts*) para estimar o valor tático de um estado. Uma estrutura de dados convencional (como matrizes de objetos ou listas de listas em Python) causaria uma sobrecarga gigantesca de alocação de memória e pesquisa na árvore de objetos, limitando severamente o número de iterações por segundo.

### 1.2 Implementação Matemática de Bitboards
Adotámos uma representação vetorial binária em que o estado do tabuleiro é comprimido em **inteiros de 64 bits** (Bitboards):
- Um bitboard para as peças do jogador **'X'**.
- Um bitboard para as peças do jogador **'O'**.
- Máscaras de colunas para gerir os limites físicos e detetar condições de inserção (*Drop*) ou remoção pela base (*Pop*).

As operações de modificação do tabuleiro foram reescritas para utilizar exclusivamente **bit-shifting aritmético e máscaras booleanas** bit a bit na CPU, eliminando ciclos de iteração por arrays.

### 1.3 Resultados Empíricos de Otimização
- **Tempo de execução para 10.000 simulações MCTS (Estrutura Clássica):** `20.0s`  
- **Tempo de execução para 10.000 simulações MCTS (Bitboards):** `0.9s`  
- **Melhoria de Performance:** **~95.5% de redução de tempo de CPU**, viabilizando pesquisas profundas e robustas em tempo real.

In [ ]:
# Demonstração conceitual da otimização estrutural de estado através de Bitboards
class BitBoardDemo:
    def __init__(self):
        self.bp_x = 0b0  # Inteiro contendo posições de 'X'
        self.bp_o = 0b0  # Inteiro contendo posições de 'O'
        self.cols_mask = [0b111111 << (i * 6) for i in range(7)]  # 7 colunas x 6 linhas
        
    def drop_piece(self, col, player):
        # Operação de Drop feita via operações lógicas bit a bit instantâneas
        current_occupied = self.bp_x | self.bp_o
        col_bits = current_occupied & self.cols_mask[col]
        # Encontra a primeira linha vaga deslocando os bits da coluna correspondente
        # Exemplo simplificado de bit-shift de performance:
        shift_pos = (col * 6) + (col_bits >> (col * 6)).bit_length()
        if player == 'X':
            self.bp_x |= (1 << shift_pos)
        else:
            self.bp_o |= (1 << shift_pos)
        return f"Peça colocada na posição de bit: {1 << shift_pos:b}"

bb = BitBoardDemo()
print(bb.drop_piece(col=3, player='X'))

---
## Capítulo 2: Algoritmo Monte Carlo Tree Search (MCTS) e as suas Variantes

O nosso motor de exploração em tempo real baseia-se no ciclo clássico de 4 passos:
1. **Seleção:** Navegação pela árvore usando o critério **UCT (Upper Confidence Bound applied to Trees)**.
2. **Expansão:** Adição de novos nós filhos legais à árvore de procura.
3. **Rollout (Simulação):** Partida simulada até ao fim do jogo para recolha de recompensa estatística.
4. **Backpropagation:** Propagação reversa dos resultados para atualizar a taxa de vitória dos nós pais.

### 2.1 Equação UCT Base
A seleção é regida pela fórmula:
$$\text{UCT} = \frac{W_i}{N_i} + C \times \sqrt{\frac{\ln(N_p)}{N_i}}$$
Onde:
- $W_i$: número de vitórias do nó filho.
- $N_i$: número de visitas do nó filho.
- $N_p$: número de visitas do nó pai.
- $C$: constante de exploração que balanceia o aproveitamento (*exploitation*) e a descoberta (*exploration*).

### 2.2 Variantes Desenvolvidas para Teste Competitivo
Desenvolvemos e parametrizámos **7 variantes independentes** do MCTS para avaliar o seu comportamento prático:
- **`UCT_base`**: Constante de exploração estável ($C = 1.414$) com rollouts puramente aleatórios.
- **`UCT_low_c`**: Redução agressiva de exploração ($C = 0.5$). Força o agente a focar estritamente nas melhores linhas táticas de vitória encontradas nas iterações iniciais.
- **`UCT_epsilon_greedy`**: Aplicação de exploração estocástica no passo de rollout com uma probabilidade $\epsilon$ de tomar jogadas heurísticas vs. aleatórias.
- **`UCT_heuristic_rollout`**: Substituição de simulações aleatórias por rollouts inteligentes orientados por regras táticas humanas (bloqueio imediato de 4-em-linha inimigo e preenchimento estratégico da coluna central).
- **`UCT_depth_limited`**: Poda estatística de simulações por limite rígido de profundidade de simulação.

---
## Capítulo 3: Pipeline de Extração de Dados e Criação de Datasets

A nossa estratégia para criar uma Inteligência Artificial estática (ID3) baseia-se na **Aprendizagem Supervisionada por Clonagem de Comportamento**. Para isso, estruturámos a geração de dados em três níveis incrementais de complexidade:

### 3.1 `generate_dataset.py` (Fase de Auto-Jogo / Prova de Conceito)
Ponto de partida clássico de extração. O agente MCTS joga contra si mesmo. A cada turno, antes de efetuar a jogada, serializa o estado complexo do tabuleiro num mapa plano de 42 atributos geométricos (`c0_r0` a `c6_r5`), correspondentes às casas da matriz. A jogada final decidida pelo MCTS serve como a etiqueta de classe (*Label*). É gerada uma base de dados limpa e equilibrada, provando a estabilidade do fluxo de I/O.

### 3.2 `generate_dataset_tournament.py` (Fase Competitiva / Dados de Elite)
Para obter dados de alta qualidade tática, colocámos as 7 variantes MCTS a combater num torneio fechado em estilo *Round-Robin* (Todos contra Todos). 
- **Mitigação do Viés do Primeiro Jogador:** Cada par de agentes defronta-se em jogos cruzados alternando quem joga com as peças 'X' (Primeiro) e 'O' (Segundo), visto que no PopOut a abertura confere vantagem geométrica.
- O dataset recolhido (`tour_craz_1.csv`) é rico em termos de competição, capturando como mentes com diferentes parâmetros reagem a ataques diversificados.

### 3.3 `generate_dataset_perturbed_tournament.py` (Fase de Robustez / Vacina contra Overfitting)
Se o modelo ID3 final fosse treinado apenas com os jogos perfeitos do torneio, ele aprenderia apenas trajetórias ideais. Em jogo real contra humanos, movimentos imprevistos ou erros crasso poderiam atirar a árvore para estados desconhecidos, causando falhas graves.
- **Data Augmentation:** O script reconstrói os tabuleiros do torneio e injeta caoticamente entre **2 a 6 jogadas completamente aleatórias (perturbações)**.
- **Rotulagem Otimizada:** Após baralhar o tabuleiro, os melhores agentes MCTS (ex: a 10.000 iterações) são injetados diretamente nessa posição confusa para calcular a jogada correta de recuperação. Isto ensina o ID3 a recuperar de desastres táticos.

In [ ]:
# Estrutura conceitual do processamento paralelo de perturbações
import random

def simulate_data_augmentation_step(base_state, mcts_expert):
    # 1. Injetar ruído controlado (simulação de 2 a 6 jogadas aleatórias humanas)
    perturbed_state = base_state.copy()
    num_perturbations = random.randint(2, 6)
    
    for _ in range(num_perturbations):
        moves = perturbed_state.get_legal_moves()
        perturbed_state.apply_move(random.choice(moves))
        if perturbed_state.is_terminal():
            return None # Descartar estados terminais acidentais
            
    # 2. O Especialista MCTS limpa o caos e dita a decisão ideal
    expert_label = mcts_expert.choose_move(perturbed_state)
    
    # Retorna o par (Atributos Geométricos, Label de Elite)
    return {"board": perturbed_state.to_features(), "classe_jogada": expert_label}
print("Pipeline de Robustez Estruturado com Sucesso.")

---
## Capítulo 4: Validação Matemática do ID3 - O Caso do Dataset Íris

Antes de aplicar o motor do ID3 ao ambiente de jogo, validámos a pureza matemática do nosso algoritmo codificado do zero utilizando o clássico dataset de flores Íris (`iris.py`).

### 4.1 O Desafio Teórico dos Dados Contínuos
O algoritmo ID3 nativo funciona estritamente com atributos categóricos/discretos (ramos bem definidos). No entanto, o dataset Íris é constituído por dimensões numéricas contínuas (ex: comprimento de sépala como `5.1`, `5.4`). Forçar ramificações para cada número real resultaria numa árvore infinitamente profunda com erro de generalização catastrófico.

### 4.2 Solução: Discretização com `pd.qcut`
Para resolver a limitação, implementámos uma camada de pré-processamento que aplica a técnica de quantis (`pd.qcut`) em cada dobra isolada de treino. Os valores são divididos em 3 categorias balanceadas em volume:
- `['Baixo', 'Medio', 'Alto']`

### 4.3 Rigor Metodológico: Estratificação e 5-Fold Cross-Validation
Para blindar os testes contra desequilíbrios estatísticos acidentais, criámos um particionamento **estratificado**, garantindo que a proporção exata das três classes biológicas (*Iris-setosa*, *Iris-versicolor*, *Iris-virginica*) se mantém estável em cada uma das 5 divisões de teste.

In [ ]:
# Simulação do pré-processamento aplicado no script iris.py
import pandas as pd
import numpy as np

# Exemplo de criação de dados contínuos simulando características da Íris
np.random.seed(42)
data_iris = pd.DataFrame({
    'sepal_length': np.random.uniform(4.3, 7.9, 10),
    'class': ['Iris-setosa']*3 + ['Iris-versicolor']*4 + ['Iris-virginica']*3
})

# Discretização em buckets balanceados por quantil (qcut)
data_iris['sepal_length_discretized'] = pd.qcut(data_iris['sepal_length'], q=3, labels=['Baixo', 'Medio', 'Alto'])
data_iris

---
## Capítulo 5: O Motor de Indução ID3 no PopOut, Poda e Regularização

A transferência do algoritmo ID3 para o PopOut exige mecânicas avançadas de controlo de complexidade estatística, desenvolvidas no script `id3_perturbed.py`.

### 5.1 Seleção Absoluta de Recursos (Clonagem de Comportamento Pura)
O dataset perturbado possui decisões cruzadas. O nosso pipeline aplica um filtro radical para isolar as respostas de um único mestre absoluto: `AGENTE_ALVO = "UCT_heuristic_rollout_10k"`. Isto remove contradições de dados que arruinariam o ganho de informação, permitindo ao ID3 convergir numa réplica exata do comportamento de topo.

### 5.2 Hiperparâmetros de Pré-Poda (Restrições de Crescimento)
Para evitar a memorização de ruído nas folhas periféricas da árvore, introduzimos bloqueios rígidos no crescimento recursivo:
- **`MAX_DEPTH = 12`**: Impede cadeias de decisão infinitamente complexas.
- **`MIN_SAMPLES_LEAF = 6`**: Um nó só pode converter-se em folha de resposta se agrupar, no mínimo, 6 estados de jogo reais. Elimina exceções estatísticas.
- **`MIN_GAIN = 0.005`**: Interrompe divisões se o ganho de informação for menor que 0.5%.

### 5.3 Algoritmo de Pós-Poda Baseado em Dados de Validação (*Validation Post-Pruning*)
Separámos **20% do dataset** exclusivamente para validação. O algoritmo constrói a árvore total e depois executa uma varredura de baixo para cima (*bottom-up*). Ele colapsa experimentalmente sub-ramos adjacentes numa folha de decisão maioritária e calcula a precisão real contra o lote oculto de 20%. Se a precisão se mantiver estável ou subir, o corte do ramo é efetuado permanentemente, eliminando o overfitting gerado pelo ruído das perturbações.

In [ ]:
# Estrutura central do cálculo recursivo com restrições e entropia
import math
from collections import Counter

def calcular_entropia(linhas, coluna_alvo):
    total = len(linhas)
    if total == 0: return 0
    contagem = Counter([r[coluna_alvo] for r in linhas])
    entropia = 0
    for v in contagem.values():
        p = v / total
        entropia -= p * math.log2(p)
    return entropia

print("Funções matemáticas de cálculo de entropia carregadas para o ID3 PopOut.")

---
## Capítulo 6: O Torneio de Árvores ID3 - Análise Crítica de Resultados

Para coroar o projeto, extraímos as árvores estáticas resultantes de diferentes treinos e colocámo-las a combater num motor competitivo autónomo (`tournament_id3_final.py`), gerando a classificação empírica final:

| Posição | Agente (Modelo ID3 Treinado) | Pontos | Histórico (V - E - D) | Estado Tático Estatístico |
| :--- | :--- | :---: | :---: | :--- |
| **1º** | **`Tour_Adv`** | **18** | 6 - 0 - 2 | **Ponto de Equilíbrio (*Sweet Spot*)** |
| **2º** | **`Tour_Craz_2_Perturbed`** | **15** | 5 - 0 - 3 | **Sucesso da Vacina (Generalização Elevada)** |
| **3º** | **`ID3_v5`** | **12** | 4 - 0 - 4 | Modelo de Referência Intermédio |
| **4º** | **`Tour_Craz_1`** | **9** | 3 - 0 - 5 | Overfitting Severo (Treino Puro de Elite) |
| **5º** | **`Tour_Base`** | **6** | 2 - 0 - 6 | *Garbage In, Garbage Out* (MCTS Fraco) |

### 6.1 Descoberta Científica: O Paradoxo da Intransitividade (Frente a Frente)
Ao testar os modelos na interface de jogo interativa (`main.py`), deparámo-nos com um fenómeno fascinante da Teoria de Jogos:
- **O Paradoxo:** O **`Tour_Craz_2_Perturbed` (2º Lugar)** derrota de forma consistente e esmagadora o **`Tour_Adv` (1º Lugar)** num confronto direto individual de 1 vs 1.

#### Explicação Científica do Fenómeno:
1. **Por que é que o `Tour_Adv` ganhou o torneio geral?** O `Tour_Adv` foi treinado com uma base intermédia (MCTS a 2.000 iterações). Ele extraiu regras muito gerais de controlo e bloqueio tático. Num torneio longo, ele é incrivelmente **consistente**: não comete erros estúpidos e pontua o máximo possível contra todos os adversários inferiores da base da tabela.
2. **Por que é que o `Tour_Craz_2_Perturbed` o destrói no confronto direto?** O `Craz_2` herda a árvore profunda de um especialista de 10.000 iterações vacinado contra o caos. Ele possui uma acuidade tática muito superior. Num duelo fechado de inteligências, ele consegue armar armadilhas estratégicas complexas a longo prazo (como forçar eixos de *pop* duplo) que o campeão geral não tem capacidade preditiva para detetar a tempo.
3. **Conclusão:** A força em sistemas complexos de Inteligência Artificial não é linear ($A > B > C$), mas sim **intransitiva** (dinâmica circular estilo Pedra-Papel-Tesoura).

---
## Capítulo 7: Interface do Utilizador, Arquitetura do Game Loop e Fallback

O ficheiro orquestrador final `main.py` consolida o projeto num software interativo, integrando os módulos através de um ciclo estruturado de controlo:

### 7.1 Arquitetura do Game Loop
O motor corre num ciclo perpétuo com quatro barreiras rígidas:
1. **Renderização Otimizada:** Tradução instantânea dos inteiros binários do Bitboard para representação visual em caracteres no terminal.
2. **Validação de Integridade de Input:** A função `get_human_move()` atua como guarda do sistema, impedindo jogadas fora de grelha ou remoções ilegais.
3. **Resolução de Ambiguidades Criticas:** O árbitro resolve os empates imediatos ou vitórias conjuntas causadas pela regra do *Pop Move* (Regra de ouro das especificações).

### 7.2 O Sistema de Paraquedas (MCTS Fallback)
O ID3 preditivo opera com base em regras discretas fixas. Caso caia numa situação absurdamente rara, ou tente tomar um movimento ilegal derivado do limite de generalização, a aplicação não falha.
- **Mecânica de Salvaguarda:** O código intercepta o estado inválido e convoca instantaneamente um **MCTS leve de segurança (200 iterações)** como fallback. O bot joga o movimento seguro e devolve o controlo ao loop, garantindo estabilidade absoluta de execução em tempo real contra utilizadores humanos.

In [ ]:
# Demonstração funcional do mecanismo de segurança e fallback em tempo real do main.py
def pick_game_move(current_state, trained_id3_tree, fallback_mcts_agent):
    try:
        # 1. Tentar classificar usando o conhecimento estático condensado no ID3
        features = {} 
        label_prediction = "drop_3"  # Exemplo simulado de retorno
        
        # Simulação de verificação de legalidade tática
        is_legal = False # Simulamos que a árvore escolheu uma jogada ilegal
        if not is_legal:
            raise ValueError("Jogada estatisticamente inválida gerada pela árvore")
            
        return label_prediction, "ID3 Puro"
        
    except Exception as e:
        # 2. Ativação do paraquedas: o MCTS assume o controlo da CPU e salva a partida
        safe_move = "drop_central" 
        return safe_move, f"MCTS Fallback acionado devido a: {str(e)}"

move, info = pick_game_move(None, None, None)
print(f"Decisão do Motor -> Jogada: {move} | Estratégia Utilizada: {info}")

---
## Conclusão Geral do Projeto

Este trabalho prático demonstra que o desenvolvimento de sistemas inteligentes bem-sucedidos vai muito além da escolha abstrata de algoritmos; reside na **Engenharia de Dados de Extrema Performance e no Pensamento Crítico**:
1. Os **Bitboards** provaram que otimização estrutural de baixo nível liberta o potencial dos algoritmos de busca.
2. O **Aumento de Dados por Perturbação** funcionou como a vacina perfeita contra o *overfitting*, transformando um classificador rígido num guerreiro resiliente.
3. O **Estudo da Intransitividade** revelou que a vitória em torneios pertence à consistência estatística, enquanto a dominância direta pertence à profundidade especializada.

O resultado final é uma arquitetura híbrida robusta, fluida, modular e teoricamente fundamentada nas melhores práticas de Ciência de Computadores.